# WeldFatigueChecker · 自采 40 张增量训练（ModelScope 魔搭 Notebook）
国内直连、无需代理。把 `raw_mine` 自采焊缝图（40 张）并入基于 120 轮权重的检测模型，采用**持续学习模式，历史数据绝不重跑**。

## 前置准备（在 modelscope.cn 网页操作）
1. 登录 modelscope.cn（淘宝 / 阿里 / 手机号均可，国内直连，无验证码墙）。
2. 进入「我的 Notebook」→ 新建实例：
   - 镜像选 **PyTorch**（或 “PyTorch + CUDA” 系列）；
   - 算力选 **GPU**（新用户通常有免费额度或低价 GPU，如 A10 / T4）；若暂时无 GPU 额度，选 CPU 也能跑（仅慢一些）。
3. 实例启动后（JupyterLab），上传两个文件到实例**工作目录**（左侧文件面板“上传”）：
   - `ml_kaggle_upload.zip`（本地 `D:\workbuddy\workbuddy学习\weld_fatigue_checker\ml_kaggle_upload.zip`，140MB，已带 `ml/` 外壳）；
   - `last.pt`（本地 `C:\Users\Administrator\Desktop\log\last.pt`，6.0M，120 权重）。
   - ⚠️ 别用 AI Studio 那个 18M 的 `WeldDefectModel.pt`。
4. 上传本 `.ipynb` 后**从上到下依次运行**所有 cell。


In [ ]:
# ModelScope 的 PyTorch 镜像已含 torch + CUDA；补装 ultralytics 与 coremltools（导出 .mlpackage 需要）
!pip install -q ultralytics pyyaml coremltools
import torch, ultralytics
print('torch', torch.__version__, '| cuda', torch.cuda.is_available(), '| ultralytics', ultralytics.__version__)


In [ ]:
import os
print('当前工作目录 :', os.getcwd())
# 解压上传的 zip（已带 ml/ 外壳）到当前目录
!unzip -q ml_kaggle_upload.zip -d .
os.chdir('ml')
ML_DIR = os.getcwd()
CKPT   = '../last.pt'   # 120 权重（上传到工作目录根，即 ml 的上级目录）
EPOCHS = 40   # 首次只训 40 张自采图：30~40 轮足够；若同时上传了源1~4全量数据，改 120
print('ML_DIR =', ML_DIR)
print('ml 目录内容 :', sorted(os.listdir('.'))[:20])
%cd {ML_DIR}


In [ ]:
import os
print('weld_train.py 存在 :', os.path.isfile('weld_train.py'))
print('raw_mine 图片数   :', len([f for f in os.listdir('raw_mine/images') if f.lower().endswith(('.jpg','.png','.jpeg'))]))
print('raw_mine 标签数   :', len([f for f in os.listdir('raw_mine/labels') if f.lower().endswith('.txt')]))
print('120 权重存在      :', os.path.isfile(CKPT))


In [ ]:
# 持续学习：首次（无 manifest）把 40 张当新增全量训一次建立基线；之后重跑只训新增，历史不重跑
!python weld_train.py --incremental --init-from {CKPT} --epochs {EPOCHS}


In [ ]:
import glob, os
runs = sorted(glob.glob('runs/detect/weld_defect_5cls*/weights/best.pt'))
print('最新训练产物      :', runs[-1] if runs else '未找到')
print('mlpackage 已导出  :', os.path.isdir('WeldDefectModel.mlpackage'))
print('已训图清单 manifest:', os.path.isfile('dataset/.trained_manifest.txt'))
if os.path.isfile('dataset/.trained_manifest.txt'):
    with open('dataset/.trained_manifest.txt') as f:
        print('  已训图数        :', len([l for l in f if l.strip()]))


In [ ]:
from IPython.display import FileLink
display(FileLink('WeldDefectModel.mlpackage'))
print('点击上面链接下载 WeldDefectModel.mlpackage；若链接无效，直接在左侧文件面板找到 ml/WeldDefectModel.mlpackage 右键下载。')
print('建议同时把 ml/runs/detect/.../weights/last.pt 下载到本机兜底（每个 epoch 已自动备份）。')


## 以后继续补充自采图（持续学习，历史绝不重跑）
1. 把新图丢进 `raw_mine/images/` + 同名标签 `raw_mine/labels/`（YOLO 5 类格式）。
2. 重跑上面的**训练 Cell**，但去掉 `--init-from`，改为：
   ```python
   !python weld_train.py --incremental
   ```
   - 脚本自动借上次 `last.pt` 权重，只训【新增图】+ 抽 ≤200 张历史图回放防遗忘，默认 30 轮，数十分钟级。
3. 下载新 `WeldDefectModel.mlpackage` 回 Xcode 工程替换即可。

⚠️ ModelScope 实例工作目录（`/mnt/data` 下）默认持久化，但仍建议每次训完把 `ml/dataset/.trained_manifest.txt` + `ml/runs/` 下载回本机 `D:\workbuddy\workbuddy学习\weld_fatigue_checker\ml\` 作为永久备份，下次重传即可续链。
⚠️ 不要把 `dataset/` 整个删除（会丢 `.trained_manifest.txt`）；想彻底重排历史才删该 manifest 文件。
